# 01. Верле: энергия, фазовая траектория и устойчивость

**Связь с лекцией:** интегрирование уравнений движения, ансамбль NVE. **Время занятия:** 20–30 минут.

Один атом движется в одномерном потенциале $U=kx^2/2$. Начальные y, z и соответствующие скорости равны нулю. Реализуем velocity Verlet явно через `CustomIntegrator`: координаты и скорости относятся к одному времени. Штатный `VerletIntegrator` использует leapfrog; не смешивайте определения скоростей двух схем.

**Предскажите:** будет ли энергия строго постоянной при конечном шаге?

**Запуск:** Colab → «Среда выполнения → Выполнить все». Выбирать GPU не нужно. Все исходные системы создаются кодом; загрузка PDB и ключи доступа не требуются. Выполняйте ячейки сверху вниз. После изменения параметров повторите зависимые ячейки.

**Обозначения:** энергии в kJ/mol, длины в nm, время в ps; 1 fs = 0.001 ps.

In [ ]:
# Установка фиксированной версии: CPU достаточно, GPU не требуется.
%pip -q install openmm==8.3.1 numpy matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openmm as mm
from openmm import unit
from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation
import platform
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True, 'font.size': 11})
rng = np.random.default_rng(2026)
# Маленькие учебные системы: Reference исключает накладные расходы GPU.
backend = mm.Platform.getPlatformByName('Reference')
R = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole/unit.kelvin)
print('OpenMM', mm.__version__, '| Python', platform.python_version(), '|', backend.getName())
# Числовые единицы ниже: nm, ps, dalton, kJ/mol, K.
def vv(dt):
    it = mm.CustomIntegrator(dt*unit.picoseconds)
    it.addComputePerDof('v', 'v+0.5*dt*f/m')
    it.addComputePerDof('x', 'x+dt*v')
    it.addComputePerDof('v', 'v+0.5*dt*f/m')
    return it

def state(ctx):
    st = ctx.getState(getPositions=True, getVelocities=True, getEnergy=True)
    x = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    v = st.getVelocities(asNumpy=True).value_in_unit(unit.nanometer/unit.picosecond)
    u = st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    return x, v, u

def movie(frames, title='', bonds=(), interval=60):
    """Небольшая XY-проекция. Показанные траектории могут быть трёхмерными."""
    a = np.asarray(frames)[::max(1, len(frames)//100)]
    fig, ax = plt.subplots(figsize=(5, 4))
    lo=a[:,:,:2].min(axis=(0,1)); hi=a[:,:,:2].max(axis=(0,1))
    pad=max(float(np.max(hi-lo))*0.15,0.1)
    ax.set(xlim=(lo[0]-pad,hi[0]+pad),ylim=(lo[1]-pad,hi[1]+pad),
           xlabel='x, nm',ylabel='y, nm',title=title,aspect='equal')
    dots=ax.scatter(a[0,:,0],a[0,:,1],s=45,c=np.arange(a.shape[1]),cmap='viridis')
    lines=[ax.plot([],[],color='gray')[0] for _ in bonds]
    def update(i):
        dots.set_offsets(a[i,:,:2])
        for line,(j,k) in zip(lines,bonds): line.set_data(a[i,[j,k],0],a[i,[j,k],1])
        return [dots]+lines
    ani=FuncAnimation(fig,update,frames=len(a),interval=interval,blit=True)
    html=ani.to_jshtml(); plt.close(fig); display(HTML(html))


## Модель и два интегратора

$\omega=\sqrt{k/m}$. Явный Эйлер обновляет x и v из старых значений. Это не полунеявный Euler–Cromer.

In [ ]:
mass, k, x0 = 12.0, 1200.0, 0.1
omega=np.sqrt(k/mass)
period=2*np.pi/omega
print('Период:',period,'ps; граница Verlet:',2/omega,'ps')
def oscillator(method,dt,duration=10):
    sys=mm.System(); sys.addParticle(mass)
    force=mm.CustomExternalForce('0.5*k*x*x'); force.addGlobalParameter('k',k)
    force.addParticle(0,[]); sys.addForce(force)
    if method=='Verlet': it=vv(dt)
    else:
        it=mm.CustomIntegrator(dt); it.addPerDofVariable('aold',0)
        it.addComputePerDof('aold','f/m')
        it.addComputePerDof('x','x+dt*v')
        it.addComputePerDof('v','v+dt*aold')
    ctx=mm.Context(sys,it,backend); ctx.setPositions([[x0,0,0]]); ctx.setVelocities([[0,0,0]])
    out=[]
    for j in range(round(duration/dt)+1):
        x,v,u=state(ctx); out.append([j*dt,x[0,0],v[0,0],u,0.5*mass*np.sum(v*v)])
        it.step(1)
    del ctx,it
    return np.array(out)
steps=[0.002,0.01,0.04]
runs={dt:oscillator('Verlet',dt) for dt in steps}
euler=oscillator('Euler',0.01,duration=3)


## Энергия и фазовый портрет

Сравниваем одинаковые физические интервалы. Ошибка нормируется на начальную энергию, а не на её текущее значение.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4))
for dt,a in runs.items():
    E=a[:,3]+a[:,4]
    ax[0].plot(a[:,0],(E-E[0])/E[0],label=f'dt={dt} ps')
    ax[1].plot(a[:,1],a[:,2],label=str(dt))
ax[0].set(xlabel='t, ps',ylabel='(E−E₀)/E₀',title='Verlet'); ax[0].legend()
ax[1].set(xlabel='x, nm',ylabel='v, nm/ps',title='Фазовый портрет')
plt.tight_layout(); plt.show()
err=np.array([np.max(np.abs((a[:,3]+a[:,4])/(a[0,3]+a[0,4])-1)) for a in runs.values()])
slope=np.polyfit(np.log(steps),np.log(err),1)[0]
print('Наклон log(error) vs log(dt):',round(slope,3),'(ожидается около 2)')
assert 1.8<slope<2.2
print('Проверка второго порядка пройдена.')
a=runs[0.01]; frames=np.zeros((len(a),1,3)); frames[:,0,0]=a[:,1]
movie(frames,'Гармонический осциллятор')


In [ ]:
# Параметры эксперимента
dt_euler = 0.01       # ps
duration_euler = 3.0  # ps

# Столбцы результата: t, x, v, U, K
euler_demo = oscillator(
    'Euler',
    dt=dt_euler,
    duration=duration_euler
)

t = euler_demo[:, 0]
x = euler_demo[:, 1]
v = euler_demo[:, 2]

# Точная фазовая траектория осциллятора для x(0)=x0, v(0)=0
phase = np.linspace(0, 2*np.pi, 500)
x_exact = x0 * np.cos(phase)
v_exact = -omega * x0 * np.sin(phase)

fig, ax = plt.subplots(1,2,figsize=(10,4))

ax[0].plot(
    x_exact, v_exact, 'k--', lw=1.5,
    label='Точное решение: постоянная энергия'
)
ax[0].plot(x, v, color='tab:orange', alpha=0.5, lw=1)
points = ax[0].scatter(x, v, c=t, cmap='plasma', s=10)

ax[0].scatter(x[0], v[0], color='green', s=70,
           marker='o', label='Начало', zorder=5)
ax[0].scatter(x[-1], v[-1], color='red', s=90,
           marker='x', label='Конец', zorder=5)

fig.colorbar(points, ax=ax[0], label='Время, ps')
ax[0].set(
    xlabel='x, nm',
    ylabel='v, nm/ps',
    title=f'Явный Эйлер: фазовая диаграмма, Δt = {dt_euler} ps'
)
ax[0].legend()

ax[1].plot(euler_demo[:,0],euler_demo[:,3]+euler_demo[:,4]); ax[1].set(xlabel='t, ps',ylabel='E, kJ/mol',title='Явный Эйлер')

plt.tight_layout()
plt.show()

# Анимация движения атома — аналогично примеру Верле
frames_euler = np.zeros((len(t), 1, 3))
frames_euler[:, 0, 0] = x

movie(
    frames_euler,
    title=f'Явный Эйлер: рост амплитуды, Δt = {dt_euler} ps'
)

## Граница устойчивости

Преднамеренно нарушаем условие ωΔt < 2. Не переносите эти большие шаги на атомистические модели.

In [ ]:
fig,ax=plt.subplots()
for h in [0.5, 1.5, 2.05]:
    a=oscillator('Verlet',h/omega,duration=4)
    ax.semilogy(a[:,0],a[:,3]+a[:,4],label=f'ωΔt={h}')
ax.set(xlabel='t, ps',ylabel='E, kJ/mol'); ax.legend(); plt.show()


## Самостоятельная работа

1. Удвойте жёсткость. Как изменились период и максимально допустимый шаг?
2. Постройте U(t) и K(t) отдельно. Почему они осциллируют даже при почти постоянной E?
3. Уменьшите шаг вдвое: во сколько раз уменьшается амплитуда ошибки?

**Ориентиры:** период уменьшается в √2 раза; энергия переходит между U и K; ошибка второго порядка уменьшается примерно в четыре раза.

## Источники

[OpenMM: интеграторы](https://docs.openmm.org/latest/userguide/theory/04_integrators.html).